# Scaling Laws 工作流

这个 notebook 是 API 快速入门的可复制、可运行版本。它展示如何查看响应示例、提交实验、轮询结果、汇总实验，并准备最终提交。

Notebook 中不会保存 API key，也不会保存执行输出。如果没有设置 `SCALING_API_BASE_URL` 和 `SCALING_API_KEY`，代码单元会使用示例响应，因此你可以在不消耗预算的情况下阅读和运行流程。

## 1. 配置 API 访问

课程团队会提供 API base URL 和个人 API key。建议在启动 Jupyter 前在 shell 中设置：

```bash
export SCALING_API_BASE_URL="https://course-api.example"
export SCALING_API_KEY="your-personal-api-key"
```

API key 是个人凭证。不要把它写进 notebook，不要提交到报告，也不要分享给其他人。

In [ ]:
import csv
import json
import os
import sys
import time
from pathlib import Path
from pprint import pprint

import requests

# 课程团队会提供这两个值。建议在启动 Jupyter 前通过环境变量设置，避免把密钥写入 notebook。
api_base_url = os.environ.get("SCALING_API_BASE_URL", "").rstrip("/")
api_key = os.environ.get("SCALING_API_KEY", "")
api_headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
LIVE_API_ENABLED = bool(api_base_url and api_key)


def api_request(method, path, **kwargs):
    """使用 plain requests 调用课程 API。"""
    if not LIVE_API_ENABLED:
        raise RuntimeError(
            "Live API disabled. 请先设置 SCALING_API_BASE_URL 和 SCALING_API_KEY。"
        )
    response = requests.request(
        method,
        f"{api_base_url}{path}",
        headers=api_headers,
        timeout=30,
        **kwargs,
    )
    try:
        body = response.json()
    except ValueError:
        body = {"raw_response": response.text}

    if response.status_code == 409:
        return body
    if response.status_code < 200 or response.status_code >= 300:
        print(
            json.dumps(body, ensure_ascii=False, indent=2, sort_keys=True),
            file=sys.stderr,
        )
        raise SystemExit(1)
    return body


def make_small_config(*, num_hidden_layers=2, hidden_size=128, num_attention_heads=1, train_tokens=4096, num_evals=1, learning_rate=3e-4):
    return {
        "model": {
            "num_hidden_layers": int(num_hidden_layers),
            "hidden_size": int(hidden_size),
            "num_attention_heads": int(num_attention_heads),
        },
        "training": {
            "train_tokens": int(train_tokens),
            "num_evals": int(num_evals),
            "learning_rate": float(learning_rate),
        },
    }


def wait_for_experiment(experiment_id, *, poll_interval_seconds=30, max_polls=None):
    terminal_statuses = {"completed", "failed", "cancelled", "system_failed"}
    polls = 0
    while True:
        experiment = api_request("GET", f"/experiment/{experiment_id}")
        polls += 1
        print("status:", experiment.get("status"))
        if experiment.get("status") in terminal_statuses:
            return experiment
        if max_polls is not None and polls >= max_polls:
            return experiment
        time.sleep(poll_interval_seconds)


if LIVE_API_ENABLED:
    print("Live API enabled.")
else:
    print("Live API disabled. 在设置凭证前将使用示例响应。")

## 2. 查看响应格式示例

下面的示例 payload 与公开 API 的响应结构一致。它们只是普通 Python 字典，因此即使离线或不想消耗预算，也可以直接查看和运行。

In [ ]:
SAMPLE_BUDGET_RESPONSE = {
    "student_id": "student-1",
    "total_seconds": 43200,
    "reserved_seconds": 0,
    "charged_seconds": 0,
    "remaining_seconds": 43200,
}

SAMPLE_SUBMIT_SUCCESS_RESPONSE = {
    "experiment_id": "exp-000001",
    "status": "queued",
    "budget_reserved_seconds": 300,
    "resource_warnings": [
        "very_low_optimizer_steps",
    ],
    "resolved_config": {
        "parameter_count_estimate": 4260096,
        "tokens_per_optimizer_step": 1024,
        "total_optimizer_steps": 4,
        "resource_warnings": [
            "very_low_optimizer_steps",
        ],
    },
}

SAMPLE_DUPLICATE_CONFIG_RESPONSE = {
    "error": "duplicate_config",
    "message": "An identical configuration was already submitted.",
    "experiment_id": "exp-000001",
}

SAMPLE_INVALID_CONFIG_RESPONSE = {
    "error": "invalid_config",
    "message": "model.hidden_size must be divisible by model.num_attention_heads",
}

SAMPLE_COMPLETED_EXPERIMENT = {
    "experiment_id": "exp-000001",
    "status": "completed",
    "validation_losses": [4.2, 3.7],
    "final_validation_loss": 3.7,
    "failure_reason": "",
    "used_runtime_seconds": 180,
    "completed_at": "2026-07-16T15:00:00Z",
    "failed_at": "",
    "resource_warnings": [],
}

SAMPLE_FAILED_EXPERIMENT = {
    "experiment_id": "exp-000002",
    "status": "failed",
    "validation_losses": [4.6, 4.1],
    "final_validation_loss": None,
    "failure_reason": "timeout",
    "used_runtime_seconds": 600,
    "completed_at": "",
    "failed_at": "2026-07-16T15:00:00Z",
    "resource_warnings": [],
}

SAMPLE_EXPERIMENTS_RESPONSE = {
    "experiments": [SAMPLE_COMPLETED_EXPERIMENT, SAMPLE_FAILED_EXPERIMENT],
}

SAMPLE_FINAL_SUBMISSION_RESPONSE = {
    "student_id": "student-1",
    "training_config": {
        "model": {"num_hidden_layers": 2, "hidden_size": 128, "num_attention_heads": 1},
        "training": {"train_tokens": 4096, "num_evals": 1, "learning_rate": 0.0003},
    },
    "predicted_final_loss": 3.25,
    "predicted_final_loss_lower": 3.18,
    "predicted_final_loss_upper": 3.36,
    "updated_at": "2026-07-16T15:00:00Z",
    "frozen_at": "",
    "frozen_by": "",
    "freeze_reason": "",
}

pprint(SAMPLE_COMPLETED_EXPERIMENT)

## 3. 查询剩余预算

预算以秒为单位返回。排队中和运行中的任务会先预留完整请求时间，直到运行完成、失败或按课程规则被取消。

In [ ]:
budget = api_request("GET", "/budget") if LIVE_API_ENABLED else SAMPLE_BUDGET_RESPONSE
budget

## 4. 构造小型 sanity-check 配置

先用短实验跑通 API 流程，并初步检查 scaling-law 假设，再投入较多预算。公开配置只有两个顶层对象：`model` 和 `training`。

In [ ]:
config = make_small_config(
    num_hidden_layers=2,
    hidden_size=128,
    train_tokens=4096,
    learning_rate=3e-4,
)
config

你也可以加入可选字段，例如 attention heads、sequence length、batch size、optimizer 和 learning-rate schedule。不支持的字段会被拒绝，这样拼写错误不会静默改变实验含义。

In [ ]:
config_with_optional_fields = {
    "model": {
        "attention_bias": False,
        "head_dim": 64,
        "hidden_size": 256,
        "intermediate_size": 1024,
        "num_attention_heads": 4,
        "num_hidden_layers": 4,
        "num_key_value_heads": 4,
        "rms_norm_eps": 1e-6,
        "rope_theta": 1_000_000,
        "tie_word_embeddings": False,
        "dtype": "bfloat16",
        "vocab_size": 50_432,
    },
    "training": {
        "train_tokens": 131072,
        "sequence_length": 1024,
        "train_batch_size": 1,
        "validation_batch_size": 1,
        "num_evals": 8,
        "learning_rate": 2e-4,
        "optimizer": "adamw",
        "lr_schedule": "cosine",
        "warmup_fraction": 0.05,
        "weight_decay": 0.01,
    },
}
config_with_optional_fields

## 5. 提交探索实验

`requested_runtime_seconds` 是本次运行预留的预算。下面的单元默认 dry-run。只有在确实准备提交真实实验时，才把 `DRY_RUN_SUBMIT` 改成 `False`。

In [ ]:
DRY_RUN_SUBMIT = True

if DRY_RUN_SUBMIT or not LIVE_API_ENABLED:
    submit_result = SAMPLE_SUBMIT_SUCCESS_RESPONSE
else:
    submit_result = api_request(
        "POST",
        "/submit",
        json={
            "config": config,
            "requested_runtime_seconds": 300,
        },
    )

submit_result

API 也可能返回结构化错误。重复提交不会额外消耗预算；无效配置会在训练开始前被拒绝。

In [ ]:
pprint(SAMPLE_DUPLICATE_CONFIG_RESPONSE)
pprint(SAMPLE_INVALID_CONFIG_RESPONSE)

## 6. 轮询直到终止状态

终止状态包括 `completed`、`failed`、`cancelled` 和 `system_failed`。上面的 `wait_for_experiment` 函数只用 plain `requests` 轮询 API，直到出现这些状态之一。

In [ ]:
experiment_id = submit_result["experiment_id"]

if DRY_RUN_SUBMIT or not LIVE_API_ENABLED:
    experiment = SAMPLE_COMPLETED_EXPERIMENT
else:
    experiment = wait_for_experiment(
        experiment_id,
        poll_interval_seconds=30,
    )

experiment

## 7. 正确理解 validation losses

`validation_losses` 是训练过程中多次验证评估得到的有序验证损失历史。它不是预测区间，也不是 `[lower, upper]`。

对于 completed 运行，`final_validation_loss` 是该实验的标量结果，并且必须等于 `validation_losses` 的最后一个值。对于 failed 运行，`validation_losses` 可能包含部分诊断值，但 `final_validation_loss` 为 `None`，这类运行不应作为 completed scaling-law 数据点使用。

In [ ]:
validation_history = experiment.get("validation_losses") or []
final_loss = experiment.get("final_validation_loss")

loss_summary = {
    "experiment_id": experiment.get("experiment_id"),
    "status": experiment.get("status"),
    "num_validation_evaluations": len(validation_history),
    "first_validation_loss": validation_history[0] if validation_history else None,
    "last_validation_loss": validation_history[-1] if validation_history else None,
    "final_validation_loss": final_loss,
    "use_as_completed_result": experiment.get("status") == "completed" and final_loss is not None,
}

if loss_summary["use_as_completed_result"]:
    assert final_loss == validation_history[-1]

loss_summary

## 8. 汇总 completed 实验

这个表可以作为拟合 scaling law 的起点。实际分析时，建议另外维护实验日志，记录每次运行的目的、假设，以及结果如何影响下一步决策。

In [ ]:
experiments = (
    api_request("GET", "/experiments")["experiments"]
    if LIVE_API_ENABLED
    else SAMPLE_EXPERIMENTS_RESPONSE["experiments"]
)

rows = []
for item in experiments:
    validation_losses = item.get("validation_losses") or []
    rows.append(
        {
            "experiment_id": item.get("experiment_id"),
            "status": item.get("status"),
            "final_validation_loss": item.get("final_validation_loss"),
            "num_eval_points": len(validation_losses),
            "best_seen_validation_loss": min(validation_losses) if validation_losses else None,
            "used_runtime_seconds": item.get("used_runtime_seconds"),
            "failure_reason": item.get("failure_reason"),
        }
    )

completed_rows = [
    row
    for row in rows
    if row["status"] == "completed" and row["final_validation_loss"] is not None
]
completed_rows

In [ ]:
best_completed = min(
    completed_rows,
    key=lambda row: row["final_validation_loss"],
    default=None,
)

WRITE_EXPERIMENT_LOG = False
if WRITE_EXPERIMENT_LOG:
    log_path = Path("scaling_laws_experiments.csv")
    fieldnames = [
        "experiment_id",
        "status",
        "final_validation_loss",
        "num_eval_points",
        "best_seen_validation_loss",
        "used_runtime_seconds",
        "failure_reason",
    ]
    with log_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {log_path}")

best_completed

## 9. 准备最终提交

最终提交会在隐藏的最终预算上运行一次。在完成分析并准备提交选定配置前，请保持 `DRY_RUN_FINAL_SUBMISSION = True`。

这里的 prediction interval 是你对隐藏最终运行结果的不确定性估计。它和 `validation_losses` 不同，后者是每个实验实际观测到的验证损失历史。

In [ ]:
final_training_config = make_small_config(
    num_hidden_layers=4,
    hidden_size=256,
    num_attention_heads=2,
    train_tokens=131072,
    learning_rate=2e-4,
)

# Replace these placeholder values with your scaling-law prediction.
predicted_final_loss = 3.25
predicted_final_loss_lower = 3.18
predicted_final_loss_upper = 3.36

DRY_RUN_FINAL_SUBMISSION = True

if DRY_RUN_FINAL_SUBMISSION or not LIVE_API_ENABLED:
    final_submission_preview = {
        "training_config": final_training_config,
        "predicted_final_loss": predicted_final_loss,
        "predicted_final_loss_lower": predicted_final_loss_lower,
        "predicted_final_loss_upper": predicted_final_loss_upper,
    }
else:
    final_submission_preview = api_request(
        "POST",
        "/final_submission",
        json={
            "training_config": final_training_config,
            "predicted_final_loss": predicted_final_loss,
            "predicted_final_loss_lower": predicted_final_loss_lower,
            "predicted_final_loss_upper": predicted_final_loss_upper,
        },
    )

final_submission_preview

## 10. 确认当前保存的最终提交

截止日期前可以多次提交，最终评测使用截止日期前最后一次有效最终提交。截止日期后，该接口会返回冻结后的记录。

In [ ]:
if DRY_RUN_FINAL_SUBMISSION or not LIVE_API_ENABLED:
    current_final_submission = SAMPLE_FINAL_SUBMISSION_RESPONSE
else:
    current_final_submission = api_request("GET", "/final_submission")

current_final_submission